In [2]:
# ============================================================
# TASK 6 - REAL-DATA NORMALIZER VALIDATION
# Amazon ML Challenge 2026
# Business Entity Resolution
#
# Purpose:
#   1. Validate production text_normalizer.py on real data
#   2. Measure positive-pair recovery
#   3. Measure negative-pair collisions
#   4. Compare production results against Task 4 baseline
#   5. Identify representations suitable for blocking/features
#
# IMPORTANT:
#   - dataset/ is READ ONLY
#   - No external APIs
#   - No production dataset modification
#   - Deterministic sampling
# ============================================================


# ============================================================
# CELL 1 - Imports
# ============================================================

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd


# ============================================================
# CELL 2 - Paths
# ============================================================

BASE_DIR = Path(
    r"D:\Degree\Hackathons\Amazon ML Challenge\student_resource"
)

DATA_DIR = BASE_DIR / "dataset"
TRAIN_DIR = DATA_DIR / "train"

SRC_DIR = BASE_DIR / "m1_work" / "src"

REPORT_DIR = BASE_DIR / "m1_work" / "reports"

OUTPUT_DIR = (
    BASE_DIR
    / "m1_work"
    / "outputs"
    / "normalizer_validation"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

sys.path.insert(
    0,
    str(SRC_DIR)
)


# ============================================================
# CELL 3 - Configuration
# ============================================================

RANDOM_SEED = 42

# Match Task 4 sampling scale.
MAX_POSITIVE_PAIRS = 500_000

MAX_NEGATIVE_PAIRS_PER_SOURCE = 500_000

# Number of detailed collision examples to save.
MAX_COLLISION_EXAMPLES = 200

# Chunk size for processing.
CHUNK_SIZE = 50_000

rng = np.random.default_rng(
    RANDOM_SEED
)

print("Base directory:")
print(BASE_DIR)

print("\nOutput directory:")
print(OUTPUT_DIR)


# ============================================================
# CELL 4 - Import production normalizer
# ============================================================

from normalization.text_normalizer import (
    normalize_text,
    compare_representations,
)


print("Production normalizer import: SUCCESS")


# ============================================================
# CELL 5 - Load training datasets
# ============================================================

def load_tsv(path):

    print(f"\nLoading: {path}")

    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        na_filter=False,
    )

    print(
        f"Rows loaded: {len(df):,}"
    )

    return df


s1 = load_tsv(
    TRAIN_DIR / "train_source1.tsv"
)

s2 = load_tsv(
    TRAIN_DIR / "train_source2.tsv"
)

s3 = load_tsv(
    TRAIN_DIR / "train_source3.tsv"
)

gt = load_tsv(
    TRAIN_DIR / "train_ground_truth.tsv"
)


# ============================================================
# CELL 6 - Validate schemas
# ============================================================

required_source_columns = {
    "entity_id",
    "business_name",
    "business_address",
    "country",
}

required_gt_columns = {
    "source1_entity_id",
    "matched_entity_ids",
}


assert required_source_columns.issubset(
    s1.columns
)

assert required_source_columns.issubset(
    s2.columns
)

assert required_source_columns.issubset(
    s3.columns
)

assert required_gt_columns.issubset(
    gt.columns
)


print("\nDataset schema validation: PASSED")


# ============================================================
# CELL 7 - Build lookups
# ============================================================

print("\nBuilding lookups...")

s1_lookup = (
    s1
    .set_index("entity_id")
    .to_dict("index")
)

s2_lookup = (
    s2
    .set_index("entity_id")
    .to_dict("index")
)

s3_lookup = (
    s3
    .set_index("entity_id")
    .to_dict("index")
)

print("Lookups created.")


# ============================================================
# CELL 8 - Parse ground truth
# ============================================================

def parse_matches(value):

    if value is None:
        return []

    value = str(value).strip()

    if not value:
        return []

    return [
        item.strip()
        for item in value.split(",")
        if item.strip()
    ]


gt_work = gt.copy()

gt_work["matched_ids"] = (
    gt_work["matched_entity_ids"]
    .map(parse_matches)
)

gt_work["match_count"] = (
    gt_work["matched_ids"]
    .map(len)
)


print(
    "\nGround-truth Source 1 entities:",
    f"{len(gt_work):,}"
)

print(
    "Ground-truth positive pairs:",
    f"{gt_work['match_count'].sum():,}"
)


# ============================================================
# CELL 9 - Expand positive pairs
# ============================================================

def expand_positive_pairs(
    gt_df,
    target_lookup,
    source_name,
):

    rows = []

    for row in gt_df.itertuples(
        index=False
    ):

        s1_id = row.source1_entity_id

        s1_record = s1_lookup.get(
            s1_id
        )

        if s1_record is None:
            continue

        for matched_id in row.matched_ids:

            target_record = target_lookup.get(
                matched_id
            )

            if target_record is None:
                continue

            rows.append({
                "source": source_name,

                "source1_entity_id": s1_id,

                "matched_entity_id": matched_id,

                "s1_name":
                    s1_record["business_name"],

                "s1_address":
                    s1_record["business_address"],

                "s1_country":
                    s1_record["country"],

                "matched_name":
                    target_record["business_name"],

                "matched_address":
                    target_record["business_address"],

                "matched_country":
                    target_record["country"],
            })

    return pd.DataFrame(rows)


print("\nExpanding S2 positives...")

positive_s2 = expand_positive_pairs(
    gt_work,
    s2_lookup,
    "S2",
)

print(
    "S2 positive pairs:",
    f"{len(positive_s2):,}"
)


print("\nExpanding S3 positives...")

positive_s3 = expand_positive_pairs(
    gt_work,
    s3_lookup,
    "S3",
)

print(
    "S3 positive pairs:",
    f"{len(positive_s3):,}"
)


# ============================================================
# CELL 10 - Combine and sample positives
# ============================================================

positive_pairs = pd.concat(
    [
        positive_s2,
        positive_s3,
    ],
    ignore_index=True,
)


print(
    "\nTotal available positive pairs:",
    f"{len(positive_pairs):,}"
)


if len(positive_pairs) > MAX_POSITIVE_PAIRS:

    positive_pairs = (
        positive_pairs
        .sample(
            n=MAX_POSITIVE_PAIRS,
            random_state=RANDOM_SEED,
        )
        .reset_index(drop=True)
    )


print(
    "Positive pairs selected:",
    f"{len(positive_pairs):,}"
)


print(
    "\nPositive pairs by source:"
)

print(
    positive_pairs["source"]
    .value_counts()
)


# ============================================================
# CELL 11 - Production representation function
# ============================================================

REPRESENTATIONS = [
    "original",
    "unicode",
    "casefold",
    "whitespace",
    "punctuation",
    "alphanumeric",
    "sorted_tokens",
    "digits",
]


def get_representation(
    value,
    representation,
):

    return normalize_text(
        value
    )[representation]


# ============================================================
# CELL 12 - Positive recovery analysis
# ============================================================

def positive_recovery(
    df,
    left_column,
    right_column,
):

    results = []

    for representation in REPRESENTATIONS:

        start = time.time()

        left_values = (
            df[left_column]
            .map(
                lambda x:
                get_representation(
                    x,
                    representation,
                )
            )
        )

        right_values = (
            df[right_column]
            .map(
                lambda x:
                get_representation(
                    x,
                    representation,
                )
            )
        )

        # Empty values must never count as a
        # recovered exact match.
        mask = (
            (left_values == right_values)
            &
            (left_values != "")
        )

        recovered = int(
            mask.sum()
        )

        total = len(df)

        recovery = (
            recovered / total * 100
            if total
            else 0
        )

        elapsed = time.time() - start

        results.append({
            "representation":
                representation,

            "total_pairs":
                total,

            "recovered_pairs":
                recovered,

            "recovery_percent":
                recovery,

            "runtime_seconds":
                elapsed,
        })

    return pd.DataFrame(
        results
    )


print(
    "\nAnalyzing positive NAME recovery..."
)

positive_name = positive_recovery(
    positive_pairs,
    "s1_name",
    "matched_name",
)


print(
    "\nAnalyzing positive ADDRESS recovery..."
)

positive_address = positive_recovery(
    positive_pairs,
    "s1_address",
    "matched_address",
)


print(
    "\nProduction NAME recovery:"
)

print(
    positive_name.to_string(
        index=False
    )
)


print(
    "\nProduction ADDRESS recovery:"
)

print(
    positive_address.to_string(
        index=False
    )
)


# ============================================================
# CELL 13 - Incremental recovery
# ============================================================

def add_incremental(
    result_df
):

    result = result_df.copy()

    result[
        "incremental_recovery_percent"
    ] = (
        result[
            "recovery_percent"
        ]
        .diff()
        .fillna(
            result[
                "recovery_percent"
            ]
        )
    )

    return result


positive_name = add_incremental(
    positive_name
)

positive_address = add_incremental(
    positive_address
)


# ============================================================
# CELL 14 - Negative ground-truth sets
# ============================================================

def build_truth_sets(
    gt_df
):

    s2_truth = {}
    s3_truth = {}

    for row in gt_df.itertuples(
        index=False
    ):

        s1_id = row.source1_entity_id

        s2_truth[s1_id] = set()
        s3_truth[s1_id] = set()

        for entity_id in row.matched_ids:

            if entity_id.startswith(
                "S2-"
            ):
                s2_truth[
                    s1_id
                ].add(entity_id)

            elif entity_id.startswith(
                "S3-"
            ):
                s3_truth[
                    s1_id
                ].add(entity_id)

    return (
        s2_truth,
        s3_truth,
    )


s2_truth, s3_truth = (
    build_truth_sets(
        gt_work
    )
)


# ============================================================
# CELL 15 - Generate negatives
# ============================================================

def generate_negative_pairs(
    target_df,
    target_lookup,
    truth_sets,
    source_name,
    number_of_pairs,
    seed,
):

    local_rng = np.random.default_rng(
        seed
    )

    s1_ids = (
        s1["entity_id"]
        .to_numpy()
    )

    target_ids = (
        target_df["entity_id"]
        .to_numpy()
    )

    rows = []

    attempts = 0

    max_attempts = (
        number_of_pairs * 20
    )

    while (
        len(rows) < number_of_pairs
        and attempts < max_attempts
    ):

        batch_size = min(
            100_000,
            number_of_pairs - len(rows),
        )

        left_ids = local_rng.choice(
            s1_ids,
            size=batch_size,
            replace=True,
        )

        right_ids = local_rng.choice(
            target_ids,
            size=batch_size,
            replace=True,
        )

        for (
            s1_id,
            target_id,
        ) in zip(
            left_ids,
            right_ids,
        ):

            attempts += 1

            if target_id in (
                truth_sets.get(
                    s1_id,
                    set()
                )
            ):
                continue

            left_record = (
                s1_lookup[s1_id]
            )

            right_record = (
                target_lookup[target_id]
            )

            rows.append({
                "source": source_name,

                "source1_entity_id":
                    s1_id,

                "matched_entity_id":
                    target_id,

                "s1_name":
                    left_record[
                        "business_name"
                    ],

                "s1_address":
                    left_record[
                        "business_address"
                    ],

                "s1_country":
                    left_record[
                        "country"
                    ],

                "matched_name":
                    right_record[
                        "business_name"
                    ],

                "matched_address":
                    right_record[
                        "business_address"
                    ],

                "matched_country":
                    right_record[
                        "country"
                    ],
            })

            if len(rows) >= number_of_pairs:
                break

    result = pd.DataFrame(rows)

    print(
        f"{source_name}: "
        f"{len(result):,} negative pairs"
    )

    return result


print(
    "\nGenerating S2 negative pairs..."
)

negative_s2 = generate_negative_pairs(
    s2,
    s2_lookup,
    s2_truth,
    "S2",
    MAX_NEGATIVE_PAIRS_PER_SOURCE,
    RANDOM_SEED,
)


print(
    "\nGenerating S3 negative pairs..."
)

negative_s3 = generate_negative_pairs(
    s3,
    s3_lookup,
    s3_truth,
    "S3",
    MAX_NEGATIVE_PAIRS_PER_SOURCE,
    RANDOM_SEED + 1,
)


negative_pairs = pd.concat(
    [
        negative_s2,
        negative_s3,
    ],
    ignore_index=True,
)


print(
    "\nTotal negative pairs:",
    f"{len(negative_pairs):,}"
)


# ============================================================
# CELL 16 - Negative collision analysis
# ============================================================

def negative_collision(
    df,
    left_column,
    right_column,
):

    results = []

    for representation in REPRESENTATIONS:

        start = time.time()

        left_values = (
            df[left_column]
            .map(
                lambda x:
                get_representation(
                    x,
                    representation,
                )
            )
        )

        right_values = (
            df[right_column]
            .map(
                lambda x:
                get_representation(
                    x,
                    representation,
                )
            )
        )

        collision_mask = (
            (left_values == right_values)
            &
            (left_values != "")
        )

        collisions = int(
            collision_mask.sum()
        )

        total = len(df)

        collision_rate = (
            collisions / total * 100
            if total
            else 0
        )

        elapsed = (
            time.time() - start
        )

        results.append({
            "representation":
                representation,

            "negative_pairs":
                total,

            "collision_pairs":
                collisions,

            "collision_percent":
                collision_rate,

            "runtime_seconds":
                elapsed,
        })

    return pd.DataFrame(
        results
    )


print(
    "\nAnalyzing negative NAME collisions..."
)

negative_name = negative_collision(
    negative_pairs,
    "s1_name",
    "matched_name",
)


print(
    "\nAnalyzing negative ADDRESS collisions..."
)

negative_address = negative_collision(
    negative_pairs,
    "s1_address",
    "matched_address",
)


print(
    "\nNegative NAME collisions:"
)

print(
    negative_name.to_string(
        index=False
    )
)


print(
    "\nNegative ADDRESS collisions:"
)

print(
    negative_address.to_string(
        index=False
    )
)


# ============================================================
# CELL 17 - Combine recovery + collision
# ============================================================

def combine_metrics(
    positive_df,
    negative_df,
    field,
):

    result = positive_df.merge(
        negative_df[
            [
                "representation",
                "collision_percent",
            ]
        ],
        on="representation",
        how="left",
    )

    result["field"] = field

    return result[
        [
            "field",
            "representation",
            "recovery_percent",
            "incremental_recovery_percent",
            "collision_percent",
            "recovered_pairs",
            "collision_pairs",
            "runtime_seconds",
        ]
    ]


name_metrics = combine_metrics(
    positive_name,
    negative_name,
    "business_name",
)

address_metrics = combine_metrics(
    positive_address,
    negative_address,
    "business_address",
)


all_metrics = pd.concat(
    [
        name_metrics,
        address_metrics,
    ],
    ignore_index=True,
)


print(
    "\nCombined validation metrics:"
)

print(
    all_metrics.to_string(
        index=False
    )
)


# ============================================================
# CELL 18 - Compare against Task 4 baseline
# ============================================================

# These are the Task 4 values from the normalization analysis.
#
# They are used only as a reproducibility/reference check.

TASK4_NAME_BASELINE = {
    "original": 4.65,
    "unicode": 4.65,
    "casefold": 10.74,
    "whitespace": 15.73,
    "punctuation": 21.65,
    "alphanumeric": 22.92,
    "sorted_tokens": 26.68,
    "digits": 95.86,
}


TASK4_ADDRESS_BASELINE = {
    "original": 2.22,
    "unicode": 2.22,
    "casefold": 7.24,
    "whitespace": 7.41,
    "punctuation": 8.28,
    "alphanumeric": 8.33,
    "sorted_tokens": 11.70,
    "digits": 62.03,
}


comparison_rows = []


for row in name_metrics.itertuples(
    index=False
):

    baseline = TASK4_NAME_BASELINE[
        row.representation
    ]

    comparison_rows.append({
        "field":
            "business_name",

        "representation":
            row.representation,

        "task4_recovery_percent":
            baseline,

        "task6_recovery_percent":
            row.recovery_percent,

        "difference_percent":
            (
                row.recovery_percent
                - baseline
            ),
    })


for row in address_metrics.itertuples(
    index=False
):

    baseline = TASK4_ADDRESS_BASELINE[
        row.representation
    ]

    comparison_rows.append({
        "field":
            "business_address",

        "representation":
            row.representation,

        "task4_recovery_percent":
            baseline,

        "task6_recovery_percent":
            row.recovery_percent,

        "difference_percent":
            (
                row.recovery_percent
                - baseline
            ),
    })


baseline_comparison = pd.DataFrame(
    comparison_rows
)


print(
    "\nTask 4 vs Task 6:"
)

print(
    baseline_comparison.to_string(
        index=False
    )
)


# ============================================================
# CELL 19 - Country-level validation
# ============================================================

def country_recovery(
    df,
    left_column,
    right_column,
):

    rows = []

    for country, group in (
        df.groupby(
            "s1_country",
            dropna=False,
        )
    ):

        for representation in REPRESENTATIONS:

            left_values = (
                group[left_column]
                .map(
                    lambda x:
                    get_representation(
                        x,
                        representation,
                    )
                )
            )

            right_values = (
                group[right_column]
                .map(
                    lambda x:
                    get_representation(
                        x,
                        representation,
                    )
                )
            )

            mask = (
                (left_values == right_values)
                &
                (left_values != "")
            )

            rows.append({
                "country":
                    country,

                "representation":
                    representation,

                "pairs":
                    len(group),

                "recovery_percent":
                    mask.mean() * 100,
            })

    return pd.DataFrame(
        rows
    )


country_name = country_recovery(
    positive_pairs,
    "s1_name",
    "matched_name",
)


country_address = country_recovery(
    positive_pairs,
    "s1_address",
    "matched_address",
)


print(
    "\nCountry-level NAME validation:"
)

print(
    country_name.to_string(
        index=False
    )
)


print(
    "\nCountry-level ADDRESS validation:"
)

print(
    country_address.to_string(
        index=False
    )
)


# ============================================================
# CELL 20 - Source-level validation
# ============================================================

def source_recovery(
    df,
    left_column,
    right_column,
):

    rows = []

    for source, group in (
        df.groupby("source")
    ):

        for representation in REPRESENTATIONS:

            left_values = (
                group[left_column]
                .map(
                    lambda x:
                    get_representation(
                        x,
                        representation,
                    )
                )
            )

            right_values = (
                group[right_column]
                .map(
                    lambda x:
                    get_representation(
                        x,
                        representation,
                    )
                )
            )

            mask = (
                (left_values == right_values)
                &
                (left_values != "")
            )

            rows.append({
                "source":
                    source,

                "representation":
                    representation,

                "pairs":
                    len(group),

                "recovery_percent":
                    mask.mean() * 100,
            })

    return pd.DataFrame(
        rows
    )


source_name = source_recovery(
    positive_pairs,
    "s1_name",
    "matched_name",
)


source_address = source_recovery(
    positive_pairs,
    "s1_address",
    "matched_address",
)


print(
    "\nSource-level NAME validation:"
)

print(
    source_name.to_string(
        index=False
    )
)


print(
    "\nSource-level ADDRESS validation:"
)

print(
    source_address.to_string(
        index=False
    )
)


# ============================================================
# CELL 21 - Identify collision examples
# ============================================================

def collect_collision_examples(
    df,
    left_column,
    right_column,
    max_examples=200,
):

    examples = []

    for representation in REPRESENTATIONS:

        left_values = (
            df[left_column]
            .map(
                lambda x:
                get_representation(
                    x,
                    representation,
                )
            )
        )

        right_values = (
            df[right_column]
            .map(
                lambda x:
                get_representation(
                    x,
                    representation,
                )
            )
        )

        mask = (
            (left_values == right_values)
            &
            (left_values != "")
        )

        subset = df.loc[
            mask,
            [
                "source",
                "source1_entity_id",
                "matched_entity_id",
                left_column,
                right_column,
            ]
        ].copy()

        if subset.empty:
            continue

        subset = subset.head(
            max_examples
        )

        subset.insert(
            0,
            "representation",
            representation,
        )

        subset[
            "normalized_value"
        ] = left_values.loc[
            subset.index
        ].values

        examples.append(
            subset
        )

    if not examples:
        return pd.DataFrame()

    return pd.concat(
        examples,
        ignore_index=True,
    )


name_collision_examples = (
    collect_collision_examples(
        negative_pairs,
        "s1_name",
        "matched_name",
        MAX_COLLISION_EXAMPLES,
    )
)


address_collision_examples = (
    collect_collision_examples(
        negative_pairs,
        "s1_address",
        "matched_address",
        MAX_COLLISION_EXAMPLES,
    )
)


# ============================================================
# CELL 22 - Save outputs
# ============================================================

positive_name.to_csv(
    OUTPUT_DIR
    / "real_positive_name_validation.csv",
    index=False,
)

positive_address.to_csv(
    OUTPUT_DIR
    / "real_positive_address_validation.csv",
    index=False,
)

negative_name.to_csv(
    OUTPUT_DIR
    / "real_negative_name_validation.csv",
    index=False,
)

negative_address.to_csv(
    OUTPUT_DIR
    / "real_negative_address_validation.csv",
    index=False,
)

all_metrics.to_csv(
    OUTPUT_DIR
    / "real_normalizer_metrics.csv",
    index=False,
)

baseline_comparison.to_csv(
    OUTPUT_DIR
    / "task4_vs_task6_comparison.csv",
    index=False,
)

country_name.to_csv(
    OUTPUT_DIR
    / "country_name_validation.csv",
    index=False,
)

country_address.to_csv(
    OUTPUT_DIR
    / "country_address_validation.csv",
    index=False,
)

source_name.to_csv(
    OUTPUT_DIR
    / "source_name_validation.csv",
    index=False,
)

source_address.to_csv(
    OUTPUT_DIR
    / "source_address_validation.csv",
    index=False,
)

name_collision_examples.to_csv(
    OUTPUT_DIR
    / "name_collision_examples.csv",
    index=False,
)

address_collision_examples.to_csv(
    OUTPUT_DIR
    / "address_collision_examples.csv",
    index=False,
)


# ============================================================
# CELL 23 - Automatic sanity checks
# ============================================================

print(
    "\nRunning sanity checks..."
)


# ------------------------------------------------------------
# Check 1 - Production normalizer must recover
# punctuation examples
# ------------------------------------------------------------

assert (
    normalize_text(
        "ABC-Pvt. Ltd."
    )["punctuation"]
    ==
    normalize_text(
        "ABC Pvt Ltd"
    )["punctuation"]
)


# ------------------------------------------------------------
# Check 2 - Sorted tokens
# ------------------------------------------------------------

assert (
    normalize_text(
        "Global Tech Solutions"
    )["sorted_tokens"]
    ==
    normalize_text(
        "Solutions Global Tech"
    )["sorted_tokens"]
)


# ------------------------------------------------------------
# Check 3 - Empty strings
# ------------------------------------------------------------

assert not (
    normalize_text("")[
        "punctuation"
    ]
)


# ------------------------------------------------------------
# Check 4 - Metrics exist
# ------------------------------------------------------------

assert len(
    all_metrics
) == len(REPRESENTATIONS) * 2


# ------------------------------------------------------------
# Check 5 - Baseline comparison
# ------------------------------------------------------------

assert len(
    baseline_comparison
) == len(REPRESENTATIONS) * 2


print(
    "All sanity checks: PASSED"
)


# ============================================================
# CELL 24 - Generate report
# ============================================================

def fmt(value):

    return f"{value:.4f}%"


report = []

report.append(
    "# Real-Data Normalizer Validation"
)

report.append("")

report.append(
    "Amazon ML Challenge 2026 — Business Entity Resolution"
)

report.append("")

report.append("## 1. Validation scope")

report.append("")

report.append(
    f"- Positive pairs analyzed: "
    f"{len(positive_pairs):,}"
)

report.append(
    f"- Negative pairs analyzed: "
    f"{len(negative_pairs):,}"
)

report.append(
    f"- Random seed: {RANDOM_SEED}"
)

report.append("")

report.append(
    "The production `text_normalizer.py` was evaluated "
    "against real training positive pairs and sampled "
    "negative pairs."
)

report.append("")

report.append(
    "## 2. Business-name validation"
)

report.append("")

report.append(
    "| Representation | Recovery | "
    "Negative Collision |"
)

report.append(
    "|---|---:|---:|"
)

for row in name_metrics.itertuples(
    index=False
):

    report.append(
        f"| `{row.representation}` | "
        f"{fmt(row.recovery_percent)} | "
        f"{fmt(row.collision_percent)} |"
    )


report.append("")

report.append(
    "## 3. Business-address validation"
)

report.append("")

report.append(
    "| Representation | Recovery | "
    "Negative Collision |"
)

report.append(
    "|---|---:|---:|"
)

for row in address_metrics.itertuples(
    index=False
):

    report.append(
        f"| `{row.representation}` | "
        f"{fmt(row.recovery_percent)} | "
        f"{fmt(row.collision_percent)} |"
    )


report.append("")

report.append(
    "## 4. Task 4 reproducibility"
)

report.append("")

report.append(
    "Task 6 recovery results were compared with the "
    "Task 4 normalization-analysis baseline."
)

report.append("")

report.append(
    "| Field | Representation | "
    "Task 4 | Task 6 | Difference |"
)

report.append(
    "|---|---|---:|---:|---:|"
)

for row in baseline_comparison.itertuples(
    index=False
):

    report.append(
        f"| {row.field} | "
        f"`{row.representation}` | "
        f"{row.task4_recovery_percent:.4f}% | "
        f"{row.task6_recovery_percent:.4f}% | "
        f"{row.difference_percent:+.4f}% |"
    )


report.append("")

report.append(
    "## 5. Interpretation"
)

report.append("")

report.append(
    "The production normalizer is considered consistent "
    "with the Task 4 analysis when the recovery values are "
    "close to the corresponding sampled baseline."
)

report.append("")

report.append(
    "Negative collision results describe the sampled "
    "negative population and are not an exhaustive estimate "
    "of all possible cross-source collisions."
)

report.append("")

report.append(
    "Representations should therefore be used as separate "
    "signals rather than treating every normalized form as "
    "a standalone identity key."
)

report.append("")

report.append(
    "## 6. Output files"
)

report.append("")

for file in sorted(
    OUTPUT_DIR.glob("*.csv")
):

    report.append(
        f"- `{file.name}`"
    )

report.append("")

report.append(
    "## 7. Status"
)

report.append("")

report.append(
    "Real-data validation completed. "
    "The results should be reviewed before exposing "
    "representations to the blocking/retrieval stage."
)


report_path = (
    REPORT_DIR
    / "REAL_DATA_NORMALIZER_VALIDATION.md"
)

report_path.write_text(
    "\n".join(report),
    encoding="utf-8",
)


# ============================================================
# CELL 25 - Final summary
# ============================================================

print("\n" + "=" * 70)
print("TASK 6 - REAL-DATA NORMALIZER VALIDATION COMPLETED")
print("=" * 70)

print(
    "\nReport:"
)

print(
    report_path
)

print(
    "\nOutput directory:"
)

print(
    OUTPUT_DIR
)

print(
    "\nGenerated files:"
)

for file in sorted(
    OUTPUT_DIR.glob("*.csv")
):

    print(
        " -",
        file.name
    )

print(
    "\nNext step:"
)

print(
    "Review REAL_DATA_NORMALIZER_VALIDATION.md "
    "before moving to blocking."
)

Base directory:
D:\Degree\Hackathons\Amazon ML Challenge\student_resource

Output directory:
D:\Degree\Hackathons\Amazon ML Challenge\student_resource\m1_work\outputs\normalizer_validation
Production normalizer import: SUCCESS

Loading: D:\Degree\Hackathons\Amazon ML Challenge\student_resource\dataset\train\train_source1.tsv
Rows loaded: 2,206,821

Loading: D:\Degree\Hackathons\Amazon ML Challenge\student_resource\dataset\train\train_source2.tsv
Rows loaded: 5,034,616

Loading: D:\Degree\Hackathons\Amazon ML Challenge\student_resource\dataset\train\train_source3.tsv
Rows loaded: 5,285,603

Loading: D:\Degree\Hackathons\Amazon ML Challenge\student_resource\dataset\train\train_ground_truth.tsv
Rows loaded: 2,206,821

Dataset schema validation: PASSED

Building lookups...
Lookups created.

Ground-truth Source 1 entities: 2,206,821
Ground-truth positive pairs: 7,638,365

Expanding S2 positives...
S2 positive pairs: 3,693,619

Expanding S3 positives...
S3 positive pairs: 3,944,746

Total av

KeyError: "['collision_pairs'] not in index"

In [3]:
# ============================================================
# CELL 17 - Combine recovery + collision
# ============================================================

def combine_metrics(
    positive_df,
    negative_df,
    field,
):
    """
    Combine positive recovery metrics with negative
    collision metrics for one field.
    """

    # --------------------------------------------------------
    # Positive metrics
    # --------------------------------------------------------

    positive_columns = [
        "representation",
        "recovery_percent",
        "incremental_recovery_percent",
        "recovered_pairs",
        "runtime_seconds",
    ]

    positive_data = positive_df[
        positive_columns
    ].copy()

    # --------------------------------------------------------
    # Negative metrics
    # --------------------------------------------------------

    negative_columns = [
        "representation",
        "collision_percent",
        "collision_pairs",
    ]

    negative_data = negative_df[
        negative_columns
    ].copy()

    # --------------------------------------------------------
    # Merge
    # --------------------------------------------------------

    result = positive_data.merge(
        negative_data,
        on="representation",
        how="left",
        validate="one_to_one",
    )

    # --------------------------------------------------------
    # Add field name
    # --------------------------------------------------------

    result["field"] = field

    # --------------------------------------------------------
    # Final column order
    # --------------------------------------------------------

    result = result[
        [
            "field",
            "representation",
            "recovery_percent",
            "incremental_recovery_percent",
            "collision_percent",
            "recovered_pairs",
            "collision_pairs",
            "runtime_seconds",
        ]
    ]

    return result


# ============================================================
# BUSINESS NAME
# ============================================================

name_metrics = combine_metrics(
    positive_name,
    negative_name,
    "business_name",
)


# ============================================================
# BUSINESS ADDRESS
# ============================================================

address_metrics = combine_metrics(
    positive_address,
    negative_address,
    "business_address",
)


# ============================================================
# COMBINE BOTH
# ============================================================

all_metrics = pd.concat(
    [
        name_metrics,
        address_metrics,
    ],
    ignore_index=True,
)


print("\n" + "=" * 70)
print("COMBINED VALIDATION METRICS")
print("=" * 70)

print(
    all_metrics.to_string(
        index=False
    )
)


COMBINED VALIDATION METRICS
           field representation  recovery_percent  incremental_recovery_percent  collision_percent  recovered_pairs  collision_pairs  runtime_seconds
   business_name       original            4.6492                        4.6492             0.0001            23246                1        15.077891
   business_name        unicode            4.6492                        0.0000             0.0001            23246                1        15.146269
   business_name       casefold           10.7378                        6.0886             0.0001            53689                1        14.408010
   business_name     whitespace           15.7344                        4.9966             0.0001            78672                1        14.777112
   business_name    punctuation           21.6466                        5.9122             0.0001           108233                1        14.511964
   business_name   alphanumeric           22.9230                      

In [4]:
# ============================================================
# CELL 17 - CORRECTED
# Combine recovery + collision
# ============================================================

def combine_metrics(
    positive_df,
    negative_df,
    field,
):

    positive_columns = [
        "representation",
        "recovery_percent",
        "incremental_recovery_percent",
        "recovered_pairs",
        "runtime_seconds",
    ]

    negative_columns = [
        "representation",
        "collision_percent",
        "collision_pairs",
    ]

    positive_data = (
        positive_df[
            positive_columns
        ].copy()
    )

    negative_data = (
        negative_df[
            negative_columns
        ].copy()
    )

    result = positive_data.merge(
        negative_data,
        on="representation",
        how="left",
        validate="one_to_one",
    )

    result["field"] = field

    result = result[
        [
            "field",
            "representation",
            "recovery_percent",
            "incremental_recovery_percent",
            "collision_percent",
            "recovered_pairs",
            "collision_pairs",
            "runtime_seconds",
        ]
    ]

    return result


# ------------------------------------------------------------
# Business name
# ------------------------------------------------------------

name_metrics = combine_metrics(
    positive_name,
    negative_name,
    "business_name",
)


# ------------------------------------------------------------
# Business address
# ------------------------------------------------------------

address_metrics = combine_metrics(
    positive_address,
    negative_address,
    "business_address",
)


# ------------------------------------------------------------
# Combined metrics
# ------------------------------------------------------------

all_metrics = pd.concat(
    [
        name_metrics,
        address_metrics,
    ],
    ignore_index=True,
)


print(
    "\n" + "=" * 70
)

print(
    "COMBINED VALIDATION METRICS"
)

print(
    "=" * 70
)

print(
    all_metrics.to_string(
        index=False
    )
)


# ============================================================
# CELL 18 - Task 4 vs Task 6
# ============================================================

TASK4_NAME_BASELINE = {
    "original": 4.65,
    "unicode": 4.65,
    "casefold": 10.74,
    "whitespace": 15.73,
    "punctuation": 21.65,
    "alphanumeric": 22.92,
    "sorted_tokens": 26.68,
    "digits": 95.86,
}


TASK4_ADDRESS_BASELINE = {
    "original": 2.22,
    "unicode": 2.22,
    "casefold": 7.24,
    "whitespace": 7.41,
    "punctuation": 8.28,
    "alphanumeric": 8.33,
    "sorted_tokens": 11.70,
    "digits": 62.03,
}


comparison_rows = []


for row in name_metrics.itertuples(
    index=False
):

    baseline = (
        TASK4_NAME_BASELINE[
            row.representation
        ]
    )

    comparison_rows.append({
        "field":
            "business_name",

        "representation":
            row.representation,

        "task4_recovery_percent":
            baseline,

        "task6_recovery_percent":
            row.recovery_percent,

        "difference_percent":
            (
                row.recovery_percent
                - baseline
            ),
    })


for row in address_metrics.itertuples(
    index=False
):

    baseline = (
        TASK4_ADDRESS_BASELINE[
            row.representation
        ]
    )

    comparison_rows.append({
        "field":
            "business_address",

        "representation":
            row.representation,

        "task4_recovery_percent":
            baseline,

        "task6_recovery_percent":
            row.recovery_percent,

        "difference_percent":
            (
                row.recovery_percent
                - baseline
            ),
    })


baseline_comparison = pd.DataFrame(
    comparison_rows
)


print(
    "\n" + "=" * 70
)

print(
    "TASK 4 vs TASK 6"
)

print(
    "=" * 70
)

print(
    baseline_comparison.to_string(
        index=False
    )
)


# ============================================================
# CELL 19 - Country-level validation
# ============================================================

def country_recovery(
    df,
    left_column,
    right_column,
):

    rows = []

    for country, group in df.groupby(
        "s1_country",
        dropna=False,
    ):

        for representation in REPRESENTATIONS:

            left_values = (
                group[left_column]
                .map(
                    lambda x:
                    get_representation(
                        x,
                        representation,
                    )
                )
            )

            right_values = (
                group[right_column]
                .map(
                    lambda x:
                    get_representation(
                        x,
                        representation,
                    )
                )
            )

            mask = (
                (left_values == right_values)
                &
                (left_values != "")
            )

            recovery = (
                mask.mean() * 100
                if len(group)
                else 0
            )

            rows.append({
                "country":
                    country,

                "representation":
                    representation,

                "pairs":
                    len(group),

                "recovered_pairs":
                    int(mask.sum()),

                "recovery_percent":
                    recovery,
            })

    return pd.DataFrame(
        rows
    )


country_name = country_recovery(
    positive_pairs,
    "s1_name",
    "matched_name",
)


country_address = country_recovery(
    positive_pairs,
    "s1_address",
    "matched_address",
)


print(
    "\n" + "=" * 70
)

print(
    "COUNTRY-LEVEL NAME VALIDATION"
)

print(
    "=" * 70
)

print(
    country_name.to_string(
        index=False
    )
)


print(
    "\n" + "=" * 70
)

print(
    "COUNTRY-LEVEL ADDRESS VALIDATION"
)

print(
    "=" * 70
)

print(
    country_address.to_string(
        index=False
    )
)


# ============================================================
# CELL 20 - Source-level validation
# ============================================================

def source_recovery(
    df,
    left_column,
    right_column,
):

    rows = []

    for source, group in df.groupby(
        "source"
    ):

        for representation in REPRESENTATIONS:

            left_values = (
                group[left_column]
                .map(
                    lambda x:
                    get_representation(
                        x,
                        representation,
                    )
                )
            )

            right_values = (
                group[right_column]
                .map(
                    lambda x:
                    get_representation(
                        x,
                        representation,
                    )
                )
            )

            mask = (
                (left_values == right_values)
                &
                (left_values != "")
            )

            recovery = (
                mask.mean() * 100
                if len(group)
                else 0
            )

            rows.append({
                "source":
                    source,

                "representation":
                    representation,

                "pairs":
                    len(group),

                "recovered_pairs":
                    int(mask.sum()),

                "recovery_percent":
                    recovery,
            })

    return pd.DataFrame(
        rows
    )


source_name = source_recovery(
    positive_pairs,
    "s1_name",
    "matched_name",
)


source_address = source_recovery(
    positive_pairs,
    "s1_address",
    "matched_address",
)


print(
    "\n" + "=" * 70
)

print(
    "SOURCE-LEVEL NAME VALIDATION"
)

print(
    "=" * 70
)

print(
    source_name.to_string(
        index=False
    )
)


print(
    "\n" + "=" * 70
)

print(
    "SOURCE-LEVEL ADDRESS VALIDATION"
)

print(
    "=" * 70
)

print(
    source_address.to_string(
        index=False
    )
)


# ============================================================
# CELL 21 - Collision examples
# ============================================================

def collect_collision_examples(
    df,
    left_column,
    right_column,
    max_examples=200,
):

    examples = []

    for representation in REPRESENTATIONS:

        left_values = (
            df[left_column]
            .map(
                lambda x:
                get_representation(
                    x,
                    representation,
                )
            )
        )

        right_values = (
            df[right_column]
            .map(
                lambda x:
                get_representation(
                    x,
                    representation,
                )
            )
        )

        mask = (
            (left_values == right_values)
            &
            (left_values != "")
        )

        subset = df.loc[
            mask,
            [
                "source",
                "source1_entity_id",
                "matched_entity_id",
                left_column,
                right_column,
            ]
        ].copy()

        if subset.empty:
            continue

        subset = subset.head(
            max_examples
        )

        subset.insert(
            0,
            "representation",
            representation,
        )

        subset[
            "normalized_value"
        ] = (
            left_values
            .loc[
                subset.index
            ]
            .values
        )

        examples.append(
            subset
        )

    if not examples:
        return pd.DataFrame()

    return pd.concat(
        examples,
        ignore_index=True,
    )


name_collision_examples = (
    collect_collision_examples(
        negative_pairs,
        "s1_name",
        "matched_name",
        MAX_COLLISION_EXAMPLES,
    )
)


address_collision_examples = (
    collect_collision_examples(
        negative_pairs,
        "s1_address",
        "matched_address",
        MAX_COLLISION_EXAMPLES,
    )
)


print(
    "\nName collision examples:",
    len(name_collision_examples)
)

print(
    "Address collision examples:",
    len(address_collision_examples)
)


if not name_collision_examples.empty:

    print(
        "\nSample NAME collisions:"
    )

    print(
        name_collision_examples
        .head(20)
        .to_string(
            index=False
        )
    )


if not address_collision_examples.empty:

    print(
        "\nSample ADDRESS collisions:"
    )

    print(
        address_collision_examples
        .head(20)
        .to_string(
            index=False
        )
    )


# ============================================================
# CELL 22 - Save outputs
# ============================================================

positive_name.to_csv(
    OUTPUT_DIR
    / "real_positive_name_validation.csv",
    index=False,
)

positive_address.to_csv(
    OUTPUT_DIR
    / "real_positive_address_validation.csv",
    index=False,
)

negative_name.to_csv(
    OUTPUT_DIR
    / "real_negative_name_validation.csv",
    index=False,
)

negative_address.to_csv(
    OUTPUT_DIR
    / "real_negative_address_validation.csv",
    index=False,
)

all_metrics.to_csv(
    OUTPUT_DIR
    / "real_normalizer_metrics.csv",
    index=False,
)

baseline_comparison.to_csv(
    OUTPUT_DIR
    / "task4_vs_task6_comparison.csv",
    index=False,
)

country_name.to_csv(
    OUTPUT_DIR
    / "country_name_validation.csv",
    index=False,
)

country_address.to_csv(
    OUTPUT_DIR
    / "country_address_validation.csv",
    index=False,
)

source_name.to_csv(
    OUTPUT_DIR
    / "source_name_validation.csv",
    index=False,
)

source_address.to_csv(
    OUTPUT_DIR
    / "source_address_validation.csv",
    index=False,
)

name_collision_examples.to_csv(
    OUTPUT_DIR
    / "name_collision_examples.csv",
    index=False,
)

address_collision_examples.to_csv(
    OUTPUT_DIR
    / "address_collision_examples.csv",
    index=False,
)


print(
    "\n" + "=" * 70
)

print(
    "OUTPUT FILES SAVED"
)

print(
    "=" * 70
)

for file in sorted(
    OUTPUT_DIR.glob("*.csv")
):

    print(
        " -",
        file.name
    )


# ============================================================
# CELL 23 - Automatic sanity checks
# ============================================================

print(
    "\nRunning sanity checks..."
)


# ------------------------------------------------------------
# Check 1 - punctuation
# ------------------------------------------------------------

assert (
    normalize_text(
        "ABC-Pvt. Ltd."
    )["punctuation"]
    ==
    normalize_text(
        "ABC Pvt Ltd"
    )["punctuation"]
)


# ------------------------------------------------------------
# Check 2 - sorted tokens
# ------------------------------------------------------------

assert (
    normalize_text(
        "Global Tech Solutions"
    )["sorted_tokens"]
    ==
    normalize_text(
        "Solutions Global Tech"
    )["sorted_tokens"]
)


# ------------------------------------------------------------
# Check 3 - empty value
# ------------------------------------------------------------

assert (
    normalize_text(
        ""
    )["punctuation"]
    == ""
)


# ------------------------------------------------------------
# Check 4 - metric count
# ------------------------------------------------------------

expected_metric_rows = (
    len(REPRESENTATIONS)
    * 2
)

assert (
    len(all_metrics)
    ==
    expected_metric_rows
)


# ------------------------------------------------------------
# Check 5 - baseline comparison
# ------------------------------------------------------------

assert (
    len(baseline_comparison)
    ==
    expected_metric_rows
)


# ------------------------------------------------------------
# Check 6 - no missing recovery
# ------------------------------------------------------------

assert not (
    all_metrics[
        "recovery_percent"
    ]
    .isna()
    .any()
)


# ------------------------------------------------------------
# Check 7 - no missing collision
# ------------------------------------------------------------

assert not (
    all_metrics[
        "collision_percent"
    ]
    .isna()
    .any()
)


print(
    "All sanity checks: PASSED"
)


# ============================================================
# CELL 24 - Generate final report
# ============================================================

def fmt(value):

    return f"{value:.4f}%"


report = []


report.append(
    "# Real-Data Normalizer Validation"
)

report.append("")

report.append(
    "Amazon ML Challenge 2026 — Business Entity Resolution"
)

report.append("")


# ------------------------------------------------------------
# Scope
# ------------------------------------------------------------

report.append(
    "## 1. Validation scope"
)

report.append("")

report.append(
    f"- Positive pairs analyzed: "
    f"{len(positive_pairs):,}"
)

report.append(
    f"- Negative pairs analyzed: "
    f"{len(negative_pairs):,}"
)

report.append(
    f"- Random seed: {RANDOM_SEED}"
)

report.append("")

report.append(
    "The production `text_normalizer.py` was evaluated "
    "against real training positive pairs and sampled "
    "negative pairs."
)

report.append("")


# ------------------------------------------------------------
# Name
# ------------------------------------------------------------

report.append(
    "## 2. Business-name validation"
)

report.append("")

report.append(
    "| Representation | Recovery | "
    "Negative Collision |"
)

report.append(
    "|---|---:|---:|"
)

for row in name_metrics.itertuples(
    index=False
):

    report.append(
        f"| `{row.representation}` | "
        f"{fmt(row.recovery_percent)} | "
        f"{fmt(row.collision_percent)} |"
    )

report.append("")


# ------------------------------------------------------------
# Address
# ------------------------------------------------------------

report.append(
    "## 3. Business-address validation"
)

report.append("")

report.append(
    "| Representation | Recovery | "
    "Negative Collision |"
)

report.append(
    "|---|---:|---:|"
)

for row in address_metrics.itertuples(
    index=False
):

    report.append(
        f"| `{row.representation}` | "
        f"{fmt(row.recovery_percent)} | "
        f"{fmt(row.collision_percent)} |"
    )

report.append("")


# ------------------------------------------------------------
# Task 4 comparison
# ------------------------------------------------------------

report.append(
    "## 4. Task 4 reproducibility"
)

report.append("")

report.append(
    "Task 6 recovery results were compared with the "
    "Task 4 normalization-analysis baseline."
)

report.append("")

report.append(
    "| Field | Representation | "
    "Task 4 | Task 6 | Difference |"
)

report.append(
    "|---|---|---:|---:|---:|"
)

for row in baseline_comparison.itertuples(
    index=False
):

    report.append(
        f"| {row.field} | "
        f"`{row.representation}` | "
        f"{row.task4_recovery_percent:.4f}% | "
        f"{row.task6_recovery_percent:.4f}% | "
        f"{row.difference_percent:+.4f}% |"
    )

report.append("")


# ------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------

report.append(
    "## 5. Interpretation"
)

report.append("")

report.append(
    "The production normalizer is considered consistent "
    "with the Task 4 analysis when recovery values are "
    "close to the corresponding sampled baseline."
)

report.append("")

report.append(
    "Negative collision results describe the sampled "
    "negative population and are not an exhaustive estimate "
    "of all possible cross-source collisions."
)

report.append("")

report.append(
    "Representations should be used as separate signals "
    "rather than treating every normalized form as a "
    "standalone identity key."
)

report.append("")


# ------------------------------------------------------------
# Additional validation
# ------------------------------------------------------------

report.append(
    "## 6. Additional validation"
)

report.append("")

report.append(
    "Country-level and source-level validation tables "
    "were generated separately as CSV outputs."
)

report.append("")


# ------------------------------------------------------------
# Files
# ------------------------------------------------------------

report.append(
    "## 7. Output files"
)

report.append("")

for file in sorted(
    OUTPUT_DIR.glob("*.csv")
):

    report.append(
        f"- `{file.name}`"
    )

report.append("")


# ------------------------------------------------------------
# Status
# ------------------------------------------------------------

report.append(
    "## 8. Status"
)

report.append("")

report.append(
    "Real-data normalizer validation completed. "
    "Results should be reviewed before exposing "
    "representations to the blocking/retrieval stage."
)


# ------------------------------------------------------------
# Write report
# ------------------------------------------------------------

report_path = (
    REPORT_DIR
    / "REAL_DATA_NORMALIZER_VALIDATION.md"
)

report_path.write_text(
    "\n".join(report),
    encoding="utf-8",
)


print(
    "\n" + "=" * 70
)

print(
    "REPORT GENERATED"
)

print(
    "=" * 70
)

print(
    report_path
)


# ============================================================
# CELL 25 - Final summary
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "TASK 6 - REAL-DATA NORMALIZER VALIDATION COMPLETED"
)

print(
    "=" * 70
)

print(
    "\nPositive pairs analyzed:",
    f"{len(positive_pairs):,}"
)

print(
    "Negative pairs analyzed:",
    f"{len(negative_pairs):,}"
)

print(
    "\nFinal report:"
)

print(
    report_path
)

print(
    "\nOutput directory:"
)

print(
    OUTPUT_DIR
)

print(
    "\nGenerated files:"
)

for file in sorted(
    OUTPUT_DIR.glob("*.csv")
):

    print(
        " -",
        file.name
    )

print(
    "\n" + "=" * 70
)

print(
    "NEXT STEP: REVIEW TASK 4 vs TASK 6"
)

print(
    "=" * 70
)


COMBINED VALIDATION METRICS
           field representation  recovery_percent  incremental_recovery_percent  collision_percent  recovered_pairs  collision_pairs  runtime_seconds
   business_name       original            4.6492                        4.6492             0.0001            23246                1        15.077891
   business_name        unicode            4.6492                        0.0000             0.0001            23246                1        15.146269
   business_name       casefold           10.7378                        6.0886             0.0001            53689                1        14.408010
   business_name     whitespace           15.7344                        4.9966             0.0001            78672                1        14.777112
   business_name    punctuation           21.6466                        5.9122             0.0001           108233                1        14.511964
   business_name   alphanumeric           22.9230                      